# ML-04 - Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sinagdevv/flyrank-ml-internship-sinagdev/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This contract uses only the authorized 30,000-row starter CSV. It defines the page-level refresh-prioritization decision without assuming access to the gated warehouse.

## 1. Unit of analysis + time window

One row represents one pseudonymized content page. `content_id` is unique at this grain; `client_id` is the grouping boundary for client-held-out validation. The starter CSV does not contain calendar date columns. Its documented observation window is a trailing 90-day aggregate ending at export time, with a separate last-30-day versus previous-30-day comparison used to derive the current trend. Therefore this starter slice supports current-state decline ranking, not an exact calendar-date claim or a future decline label.

In [1]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "data/raw/content_refresh_anonymized.csv").exists():
    repo_root = (repo_root / "../..").resolve()
data_path = repo_root / "data/raw/content_refresh_anonymized.csv"
assert data_path.exists(), f"Starter CSV not found: {data_path}"
df = pd.read_csv(data_path)
required = {"content_id", "client_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate", "trend_direction", "trend_pct"}
assert required <= set(df.columns)
print(f"Loaded {len(df):,} starter rows and {df['client_id'].nunique():,} clients.")
print("Documented window: trailing 90-day aggregates ending at export time; exact calendar dates are not present in this CSV.")

Loaded 30,000 starter rows and 32 clients.
Documented window: trailing 90-day aggregates ending at export time; exact calendar dates are not present in this CSV.


## 2. Fields: feature / label / context / excluded

**Candidate features:** `impressions_90d`, `days_since_last_update`, `avg_position`, `ctr`, and `engagement_rate`. Each is available at the ranking decision time in this snapshot: exposure, freshness, search position, click-through rate, and engagement are all measured before a human review decision.

**Label / current-state proxy:** `trend_direction == "down"`, derived from the 30-day comparison; it describes observed current decline, not a future outcome.

**Context:** `content_id` identifies the page for joins and output, and `client_id` defines the grouping boundary for validation. Neither is a model feature.

**Excluded:** `trend_direction` and `trend_pct` because they define or directly encode the outcome; any future-window fields or post-decision measurements because they would leak information. Missing values are retained as missingness to inspect rather than blindly filled with zero.

In [2]:
candidate_features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]
forbidden_features = {"content_id", "client_id", "trend_direction", "trend_pct"}
assert not forbidden_features.intersection(candidate_features)
feature_table = pd.DataFrame({
    "feature": candidate_features,
    "refresh_prioritization_role": [
        "Visibility: limits review to pages with enough exposure to matter.",
        "Freshness: highlights pages that may merit a refresh review.",
        "Search position: indicates current search visibility and room for improvement.",
        "CTR: indicates how often visible results earn clicks.",
        "Engagement rate: indicates downstream visit quality where GA4 is available.",
    ],
    "available_at_decision": [True, True, True, True, True],
    "missing_values": [int(df[c].isna().sum()) for c in candidate_features],
})
print("Feature contract: five candidate features; IDs and outcome-derived fields excluded.")
display(feature_table)

Feature contract: five candidate features; IDs and outcome-derived fields excluded.


feature,refresh_prioritization_role,available_at_decision,missing_values
impressions_90d,Visibility: limits review to pages with enough exposure to matter.,True,0
days_since_last_update,Freshness: highlights pages that may merit a refresh review.,True,0
avg_position,Search position: indicates current search visibility and room for improvement.,True,0
ctr,CTR: indicates how often visible results earn clicks.,True,0
engagement_rate,Engagement rate: indicates downstream visit quality where GA4 is available.,True,0


## 3. Verify it with queries (grain, counts, missing values, windows)

The next cell runs exactly three checks against the starter data: grain and counts, available temporal coverage, and the documented availability flag semantics.

In [3]:
# Check 1: row counts, unique page grain, and client boundary.
duplicate_pages = int(df["content_id"].duplicated().sum())
assert duplicate_pages == 0
print(f"Check 1 - rows={len(df):,}; unique pages={df['content_id'].nunique():,}; unique clients={df['client_id'].nunique():,}; duplicate content_id rows={duplicate_pages}")

# Check 2: actual temporal coverage available in this source.
date_columns = [c for c in df.columns if c.lower().endswith("_date") or c.lower().endswith("_at") or "timestamp" in c.lower()]
assert date_columns == []
print("Check 2 - calendar date columns: none; available coverage is documented as trailing 90-day aggregates ending at export time, with last-30d and previous-30d comparison windows.")

# Check 3: documented IS TRUE availability filter where relevant.
availability_field = "ga4_data_available"
if availability_field in df.columns:
    available = df[availability_field].fillna(False).eq(True)  # pandas equivalent of SQL IS TRUE
    print(f"Check 3 - {availability_field} IS TRUE rows={int(available.sum()):,}")
else:
    print("Check 3 - ga4_data_available is not present in the starter CSV; no IS TRUE availability filter is applicable to this source.")

Check 1 - rows=30,000; unique pages=30,000; unique clients=32; duplicate content_id rows=0
Check 2 - calendar date columns: none; available coverage is documented as trailing 90-day aggregates ending at export time, with last-30d and previous-30d comparison windows.
Check 3 - ga4_data_available is not present in the starter CSV; no IS TRUE availability filter is applicable to this source.


## 4. Data limits

This starter slice is a 90-day snapshot, not a longitudinal panel with a future outcome window. It can support a current observed-decline ranking experiment, but it cannot establish whether a page will decline later or whether a refresh caused recovery. The 30-day comparison overlaps the same snapshot, so `trend_direction` and `trend_pct` are label sources, never features. Missingness can follow `content_type`; a blind zero fill could encode category membership. For warehouse work, history is unbalanced by client, GA4/GSC availability flags can be three-valued, and feature and label windows must be aligned before using `IS TRUE` filters or query tables. Human review remains required.

In [4]:
assert df["content_id"].is_unique
assert set(candidate_features).isdisjoint(forbidden_features)
assert all(feature_table["available_at_decision"])
print("Leakage check: PASS - content_id/client_id are context only; trend_direction/trend_pct are excluded outcome-derived fields; no future fields are used.")
print("Decision output: an interpretable page-priority score for human review, not an automatic refresh decision.")

Leakage check: PASS - content_id/client_id are context only; trend_direction/trend_pct are excluded outcome-derived fields; no future fields are used.
Decision output: an interpretable page-priority score for human review, not an automatic refresh decision.


## Self-check

- [x] One row, grouping boundary, and available observation window are defined.
- [x] Current decline proxy and its leakage exclusions are explicit.
- [x] Exactly three meaningful verification checks are executed.
- [x] Five candidate features include refresh relevance and decision-time availability.
- [x] At least one concrete limitation is documented; no future or causal claim is made.
- [ ] Commit and submit to the internship portal (intentionally not performed in this task).